# Treino e avaliação do Extrator Forense no Colab

Guia completo: `.docs/GUIA_COLAB_COMPLETO.md`.

Antes de começar:
1. Envie `trabalho-llm.zip` para a raiz do **Meu Drive**.
2. Ative a GPU em **Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU**.
3. Execute as células abaixo em ordem, uma de cada vez.

## 1. Confirmar a GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU T4 antes de continuar.')
print('GPU:', torch.cuda.get_device_name(0))

## 2. Montar o Drive e extrair o projeto

Extrai `Meu Drive/trabalho-llm.zip` em `Meu Drive/trabalho-llm`. Se você enviar um ZIP novo, esta célula atualiza o código sem apagar `outputs/`.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/trabalho-llm')
ZIP = Path('/content/drive/MyDrive/trabalho-llm.zip')

if ZIP.exists():
    with zipfile.ZipFile(ZIP) as arquivo:
        arquivo.extractall(BASE)
    print('Projeto extraído de', ZIP)

# Aceita ZIP com ou sem uma pasta interna (ex.: trabalho-llm/trabalho-llm/requirements.txt).
candidatos = [p.parent for p in BASE.rglob('requirements.txt')
              if '.venv' not in p.parts and (p.parent / 'src' / 'train' / 'train_qlora.py').exists()]
if not candidatos:
    raise FileNotFoundError(f'Projeto não encontrado em {BASE}. Confira se o ZIP contém requirements.txt e a pasta src/.')
PROJETO = min(candidatos, key=lambda p: len(p.parts))
os.chdir(PROJETO)
print('Pasta do projeto:', PROJETO)
!ls

## 3. Instalar dependências

Leva alguns minutos. Se no final o Colab pedir para **reiniciar a sessão**, aceite, execute de novo a célula 2 e siga para a célula 4 (não reinstale).

In [ ]:
# Usa o PyTorch que já vem no Colab (requirements.txt é fixado para Python 3.12).
!python --version
!pip install -q -r requirements-colab.txt
!python -c "import torch, transformers, peft, trl, bitsandbytes; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| transformers', transformers.__version__, '| bitsandbytes', bitsandbytes.__version__)"

## 4. Gerar e validar o dataset e rodar os testes

In [ ]:
!python src/data/generate_dataset.py --seed 42 --output data/raw/dataset.jsonl
!python src/data/split_dataset.py --input data/raw/dataset.jsonl --output-dir data/processed --seed 42
!python src/data/validate_dataset.py
!python -m pytest -q

## 5. Treinar (r=8 e r=16)

O log deve mostrar `Modo de treino: qlora`. O adaptador final em `outputs/adapter/` é o de r=16.

In [ ]:
!python src/train/train_qlora.py --config configs/train.yaml --compare-hparams
!cat outputs/curves/hparams.md

## 6. Ver as curvas de perda

In [ ]:
from IPython.display import Image, display
for nome in ('loss_r8.png', 'loss_r16.png'):
    display(Image(filename=f'outputs/curves/{nome}'))

## 7. Avaliação cega Base vs Fine-Tuned

In [ ]:
!python src/eval/run_inference.py --test data/processed/test.jsonl --adapter outputs/adapter
!python src/eval/compare.py
!cat outputs/eval/comparison.md | head -n 14

## 8. Teste com a planilha CSV (Base e Fine-Tuned)

In [ ]:
!python src/eval/evaluate_csv.py --csv data/samples/planilha_teste.csv
!python src/eval/evaluate_csv.py --csv data/samples/planilha_teste.csv --adapter outputs/adapter

## 9. Baixar os resultados

In [ ]:
!zip -r fine_tuned_outputs.zip outputs/adapter outputs/curves outputs/eval REPORT.md -x "outputs/adapter/checkpoint-*"
from google.colab import files
files.download('fine_tuned_outputs.zip')